# 02 Embeddings And Pairs
Chars2Vec/SPECTER erzeugen oder laden, Embedding-QC, LSPO-Pairs aufbauen (inkl. Leakage-Check).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sys

RUN_STAGE = "smoke"
RUN_ID = "replace_with_run_id_from_00"
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
USE_STUB_EMBEDDINGS = False
PREFER_PRECOMPUTED_ADS = True

In [ ]:
from src.common.config import load_yaml
from src.common.io_schema import read_parquet, save_parquet
from src.features.embed_chars2vec import get_or_create_chars2vec_embeddings
from src.features.embed_specter import get_or_create_specter_embeddings
from src.approaches.nand.build_pairs import assign_lspo_splits, build_pairs_within_blocks, write_pairs

model_cfg = load_yaml(PROJECT_ROOT / "configs/model/nand_best.yaml")
run_cfg = load_yaml(PROJECT_ROOT / f"configs/runs/{RUN_STAGE}.yaml")

subset_dir = PROJECT_ROOT / "data/subsets/cache" / RUN_ID
emb_dir = PROJECT_ROOT / "artifacts/embeddings" / RUN_ID
metrics_dir = PROJECT_ROOT / "artifacts/metrics" / RUN_ID
emb_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")


In [ ]:
# Embeddings erzeugen/laden
rep = model_cfg["representation"]

lspo_chars = get_or_create_chars2vec_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
lspo_text = get_or_create_specter_embeddings(
    mentions=lspo_mentions,
    output_path=emb_dir / f"lspo_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=16,
    prefer_precomputed=False,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

ads_chars = get_or_create_chars2vec_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy",
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)
ads_text = get_or_create_specter_embeddings(
    mentions=ads_mentions,
    output_path=emb_dir / f"ads_specter_{RUN_STAGE}.npy",
    model_name=rep.get("text_model_name", "allenai/specter"),
    max_length=int(rep.get("max_length", 256)),
    batch_size=32,
    prefer_precomputed=PREFER_PRECOMPUTED_ADS,
    use_stub_if_missing=USE_STUB_EMBEDDINGS,
)

print("lspo chars/text:", lspo_chars.shape, lspo_text.shape)
print("ads chars/text:", ads_chars.shape, ads_text.shape)


In [ ]:
# Embedding-QC

def emb_qc(name, arr):
    norms = np.linalg.norm(arr, axis=1)
    return {
        "name": name,
        "shape": tuple(arr.shape),
        "nan_count": int(np.isnan(arr).sum()),
        "norm_mean": float(np.mean(norms)),
        "norm_p01": float(np.quantile(norms, 0.01)),
        "norm_p99": float(np.quantile(norms, 0.99)),
    }

qc = pd.DataFrame([
    emb_qc("lspo_chars", lspo_chars),
    emb_qc("lspo_text", lspo_text),
    emb_qc("ads_chars", ads_chars),
    emb_qc("ads_text", ads_text),
])
qc


In [ ]:
# LSPO splits + pairs (label/split-aware)
lspo_mentions = assign_lspo_splits(lspo_mentions, seed=int(run_cfg.get("seed", 11)))

lspo_pairs = build_pairs_within_blocks(
    mentions=lspo_mentions,
    max_pairs_per_block=run_cfg.get("max_pairs_per_block"),
    seed=int(run_cfg.get("seed", 11)),
    require_same_split=True,
    labeled_only=False,
    balance_train=True,
)

lspo_pairs_path = subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet"
write_pairs(lspo_pairs, lspo_pairs_path)
save_parquet(lspo_mentions, subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet", index=False)

print("LSPO pairs:", len(lspo_pairs), "->", lspo_pairs_path)


In [ ]:
# ADS inference-pairs (ohne Labels)
ads_pairs = build_pairs_within_blocks(
    mentions=ads_mentions,
    max_pairs_per_block=run_cfg.get("max_pairs_per_block"),
    seed=int(run_cfg.get("seed", 11)),
    require_same_split=False,
    labeled_only=False,
    balance_train=False,
)
ads_pairs_path = subset_dir / f"ads_pairs_{RUN_STAGE}.parquet"
write_pairs(ads_pairs, ads_pairs_path)
print("ADS pairs:", len(ads_pairs), "->", ads_pairs_path)


In [ ]:
# Pair-QC inkl. Leakage-Check
q = lspo_pairs.copy()
if "label" in q.columns:
    known = q[q["label"].notna()]
    pos = int((known["label"] == 1).sum())
    neg = int((known["label"] == 0).sum())
    ratio = pos / max(1, neg)
    print({"labeled_pairs": len(known), "pos": pos, "neg": neg, "pos_neg_ratio": ratio})

pairs_per_block = q.groupby("block_key").size().describe()
print("Pairs per block:")
print(pairs_per_block)

# Leakage-Check: gleiche ORCID über split-Grenzen?
leak = 0
if "orcid" in lspo_mentions.columns and "split" in lspo_mentions.columns:
    g = lspo_mentions[lspo_mentions["orcid"].notna()].groupby("orcid")["split"].nunique()
    leak = int((g > 1).sum())
print("orcid leakage groups:", leak)

with (metrics_dir / "02_pairs_qc.json").open("w", encoding="utf-8") as f:
    json.dump({"orcid_leakage_groups": leak, "lspo_pairs": int(len(lspo_pairs)), "ads_pairs": int(len(ads_pairs))}, f, indent=2)
